In [1]:
import sys
import optuna
import numpy as np
import pandas as pd

sys.path.append("..")
from databricks_connector import get_table

pd.set_option('display.max_columns', None)
target_col = "btts"
odds_col: str = "goalNoGoal_quote_currentGG"


features = [
 'goalNoGoal_chance_goal',
 'goalNoGoal_chance_goalHome',
 'goalNoGoal_chance_goalAway',
 'goalNoGoal_multigoal_m13',
 'goalNoGoal_multigoal_m14',
 'goalNoGoal_multigoal_m24',
 'goalNoGoal_multigoal_m13Home',
 'goalNoGoal_multigoal_m13Away',
 'goalNoGoal_multigoal_m24Home',
 'goalNoGoal_multigoal_m24Away',
 'goalNoGoal_quote_realGG',
 'goalNoGoal_quote_initialGG',
 'goalNoGoal_quote_initialNG',
 'goalNoGoal_quote_currentGG',
 'goalNoGoal_quote_currentNG',
 'goalNoGoal_quote_diffRealCurrGG',
 'goalNoGoal_quote_diffRealCurrNG',
 'goalNoGoal_quote_diffInitialCurrGG',
 'goalNoGoal_quote_diffInitialCurrNG',
 'goalNoGoal_comparison_affini',
 'goalNoGoal_comparison_flashback',
 'goalNoGoal_stats_avgGoalHome',
 'goalNoGoal_stats_avgGoalTakenHome',
 'goalNoGoal_stats_avgGoalAway',
 'goalNoGoal_stats_avgGoalTakenAway',
 'goalNoGoal_flashback_goal',
 'goalNoGoal_flashback_m13',
 'goalNoGoal_flashback_m24',
 'goalNoGoal_flashback_m35',
 'underOver_chance_over05HT',
 'underOver_chance_over052HT',
 'underOver_chance_over15HT',
 'underOver_chance_over15',
 'underOver_chance_over25',
 'underOver_chance_over35',
 'underOver_chance_over45',
 'underOver_quote_realO',
 'underOver_quote_initialU',
 'underOver_quote_initialO',
 'underOver_quote_currentU',
 'underOver_quote_currentO',
 'underOver_quote_diffRealCurrU',
 'underOver_quote_diffRealCurrO',
 'underOver_quote_diffInitialCurrU',
 'underOver_quote_diffInitialCurrO',
 'underOver_comparison_affini',
 'underOver_comparison_flashback',
 'underOver_flashback_under05HT',
 'underOver_flashback_over05HT',
 'underOver_flashback_under15',
 'underOver_flashback_over15',
 'underOver_flashback_under25',
 'underOver_flashback_over25',
 'underOver_flashback_under35',
 'underOver_flashback_over35',
 'evaluation_valScala',
 'evaluation_valMetrica',
 'chance1x2_quote_current1',
 'chance1x2_quote_current2'
 ]


features = ["underOver_chance_over15HT",
"chance1x2_quote_current1",
"chance1x2_quote_current2",
"goalNoGoal_quote_currentGG",
"goalNoGoal_chance_goal",
"goalNoGoal_flashback_goal",]
# "goalNoGoal_stats_avgGoalHome",
# "goalNoGoal_stats_avgGoalTakenAway",
# "goalNoGoal_stats_avgGoalTakenHome",
# "goalNoGoal_stats_avgGoalAway"]

In [2]:
# TODO: train/test split
# TODO: far scegliere a Optuna se tenere o meno una variabile
# TODO: aggiungere early stopping
# TODO: aggiungere mean e std col dato aggregato settimanalmente

In [3]:
# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """

df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

df = df_loaded.copy()

features_mapper = {x: str(df.dtypes[x]) for x in features}

def get_return(strategy: str, odds: str):
    if strategy:
        gain = odds - 1
    else:
        gain = -1
    return gain

df["return"] = df.apply(lambda x: get_return(strategy=x["btts"], odds=x["goalNoGoal_quote_currentGG"]), axis=1)

In [4]:
# Binning
df_binned = df.copy()

for feat in features:
    df_binned[feat] = pd.qcut(
        df_binned[feat],
        q=30,
        labels=False,
        duplicates="drop"
    )

In [5]:
target_col = "return"
min_obs = 50
low_cardinality_threshold = 10  # <= 10 valori unici => tratto come discreta ordinata


def objective(trial):
    mask = pd.Series(True, index=df_binned.index)

    for feat in features:
        s = df_binned[feat]

        # salta feature non numeriche
        if not pd.api.types.is_numeric_dtype(s):
            continue

        non_null = s.dropna()
        if non_null.empty:
            continue

        nunique = non_null.nunique()

        # se vuoi escludere sempre i missing
        include_missing = False

        # ---------------------------------
        # CASO 1: numerica discreta ordinata
        # ---------------------------------
        if nunique <= low_cardinality_threshold:
            unique_vals = sorted(non_null.unique().tolist())

            min_idx = trial.suggest_int(f"{feat}_min_idx", 0, len(unique_vals) - 1)
            max_idx = trial.suggest_int(f"{feat}_max_idx", min_idx, len(unique_vals) - 1)

            feat_min = unique_vals[min_idx]
            feat_max = unique_vals[max_idx]

            feat_mask = s.between(feat_min, feat_max)

            if include_missing:
                feat_mask = feat_mask | s.isna()
            else:
                feat_mask = feat_mask & s.notna()

        # ---------------------------------
        # CASO 2: numerica continua/intera
        # ---------------------------------
        else:
            if pd.api.types.is_integer_dtype(s):
                feat_min = trial.suggest_int(
                    f"{feat}_min",
                    int(non_null.min()),
                    int(non_null.max())
                )
                feat_max = trial.suggest_int(
                    f"{feat}_max",
                    feat_min,
                    int(non_null.max())
                )
            else:
                feat_min = trial.suggest_float(
                    f"{feat}_min",
                    float(non_null.min()),
                    float(non_null.max())
                )
                feat_max = trial.suggest_float(
                    f"{feat}_max",
                    feat_min,
                    float(non_null.max())
                )

            feat_mask = s.between(feat_min, feat_max)

            if include_missing:
                feat_mask = feat_mask | s.isna()
            else:
                feat_mask = feat_mask & s.notna()

        mask &= feat_mask

    selected = df_binned.loc[mask]

    if len(selected) < min_obs:
        return -1e9

    mean_return = selected[target_col].mean()
    coverage_penalty = np.log1p(len(selected))

    score = mean_return * coverage_penalty
    return float(score)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=500)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-03-26 17:41:37,651] A new study created in memory with name: no-name-c1aa7aeb-d63b-48b6-8cde-a5f62314dc9d
[I 2026-03-26 17:41:37,658] Trial 0 finished with value: -1000000000.0 and parameters: {'underOver_chance_over15HT_min': 12, 'underOver_chance_over15HT_max': 28, 'chance1x2_quote_current1_min': 25, 'chance1x2_quote_current1_max': 27, 'chance1x2_quote_current2_min': 8, 'chance1x2_quote_current2_max': 19, 'goalNoGoal_quote_currentGG_min': 14, 'goalNoGoal_quote_currentGG_max': 14, 'goalNoGoal_chance_goal_min': 20, 'goalNoGoal_chance_goal_max': 21, 'goalNoGoal_flashback_goal_min': 14.184727309084332, 'goalNoGoal_flashback_goal_max': 21.009622094439642}. Best is trial 0 with value: -1000000000.0.
[I 2026-03-26 17:41:37,662] Trial 1 finished with value: -1000000000.0 and parameters: {'underOver_chance_over15HT_min': 14, 'underOver_chance_over15HT_max': 28, 'chance1x2_quote_current1_min': 20, 'chance1x2_quote_current1_max': 22, 'chance1x2_quote_current2_min': 2, 'chance1x2_quote_c

Best params: {'underOver_chance_over15HT_min': 20, 'underOver_chance_over15HT_max': 29, 'chance1x2_quote_current1_min': 10, 'chance1x2_quote_current1_max': 28, 'chance1x2_quote_current2_min': 8, 'chance1x2_quote_current2_max': 29, 'goalNoGoal_quote_currentGG_min': 3, 'goalNoGoal_quote_currentGG_max': 3, 'goalNoGoal_chance_goal_min': 13, 'goalNoGoal_chance_goal_max': 29, 'goalNoGoal_flashback_goal_min': 17.156535981998736, 'goalNoGoal_flashback_goal_max': 28.504273358985834}
Best score: 1.0764378371486465


In [ ]:
mask = pd.Series(True, index=df_binned.index)

for key, value in study.best_params.items():
    feat = "_".join(key.split("_")[:-1])

    if key.endswith("_min"):
        mask &= (df_binned[feat] >= value).fillna(False)
    elif key.endswith("_max"):
        mask &= (df_binned[feat] <= value).fillna(False)
    else:
        raise ValueError(key)

df_filtered = df.loc[mask].copy()

mean_roi = df_filtered["return"].mean()
right = df_filtered["btts"].sum()  
total = df_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: 0.26081967213114765 (std = 0.6291271601246902)
Accuracy: 0.8 (49/61)


In [99]:
def apply_best_params(df, best_params):
    mask = pd.Series(True, index=df.index)

    # ricavo i nomi base delle feature dai best_params
    features = set()
    for k in best_params:
        if k.endswith("_min_idx"):
            features.add(k[:-8])   # rimuove "_min_idx"
        elif k.endswith("_max_idx"):
            features.add(k[:-8])   # rimuove "_max_idx"
        elif k.endswith("_min"):
            features.add(k[:-4])   # rimuove "_min"
        elif k.endswith("_max"):
            features.add(k[:-4])   # rimuove "_max"

    for feat in features:
        s = df[feat]
        non_null = s.dropna()

        if non_null.empty:
            continue

        # caso discreto ordinato: min_idx / max_idx
        if f"{feat}_min_idx" in best_params and f"{feat}_max_idx" in best_params:
            unique_vals = sorted(non_null.unique().tolist())

            min_idx = best_params[f"{feat}_min_idx"]
            max_idx = best_params[f"{feat}_max_idx"]

            feat_min = unique_vals[min_idx]
            feat_max = unique_vals[max_idx]

        # caso numerico standard: min / max
        elif f"{feat}_min" in best_params and f"{feat}_max" in best_params:
            feat_min = best_params[f"{feat}_min"]
            feat_max = best_params[f"{feat}_max"]

        else:
            continue

        mask &= s.notna() & s.between(feat_min, feat_max)

    return df.loc[mask].copy()

filtered_df = apply_best_params(df_binned, study.best_params)

roi = filtered_df["return"].mean()
right = filtered_df["btts"].sum()  
total = filtered_df.shape[0]

accuracy = np.round((right / total), 1)
print(f"ROI: {roi}")
print(f"Accuracy: {accuracy}")


ROI: 0.33359375
Accuracy: 0.8


In [102]:
right

np.int64(51)